# Animal Detection Pipeline

This project implements a **two-stage pipeline for animal detection** using Natural Language Processing and Computer Vision.

The system combines:

- **NER model (pretrained DistilBERT)** — extracts animal names from text
- **CV model (ResNet18)** — classifies animals in images

Together, these models form a unified pipeline that can process both **text and images**.

#### Install dependencies

In [1]:
%pip install -r requirements.txt

  Using cached torch-2.10.0-2-cp312-none-macosx_11_0_arm64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.4 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-4.7.0-py3-none-any.whl.metadata (19 kB)
  Using cached seqeval-1.2.2-py3-none-any.whl
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached notebook-7.5.5-py3-none-any.whl.metadata (10 kB)
  Using cached pillow-12.1.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached kaggle-2.0.0-py3-none-any.whl.metadata (15 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached scikit_lear

In [10]:
import warnings
warnings.filterwarnings("ignore")

<h1 align="center">Named Entity Recognition (NER)</h1>

At this stage, a **DistilBERT-based NER model** is used to detect animal names in text.

DistilBERT was chosen because it is a lightweight and efficient transformer model that preserves most of BERT's performance while being faster and requiring less computational resources.

In this project, DistilBERT is fine-tuned for token classification to recognize animal entities.  
A small synthetic dataset of sentences mentioning animals was generated and used to train the model.

The trained model labels each token as either:

- **O** — not an animal  
- **ANIMAL** — part of an animal name

<h2 align="center">Dataset Setup</h2>

Before training the NER model, we generate a small synthetic dataset of sentences containing animal names.

In [2]:
import subprocess
from src.utils.dataset_loader import load_dataset_for_ner

subprocess.run(["python", "src/utils/dataset_generator.py"], check=True)

# Load datasets and remove unnecessary columns
train_dataset = load_dataset_for_ner("data/ner/train.json").remove_columns(["label", "has_animal"])
val_dataset = load_dataset_for_ner("data/ner/validation.json").remove_columns(["label", "has_animal"])

train_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 169
})

Start model training

In [3]:
from src.ner.train import train_ner

trainer = train_ner(train_dataset, val_dataset)

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/sofiiapelenska/project/ML-tasks/Task 2/testTask2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.355844
20,0.072149
30,0.005879
40,0.003917
50,0.001169


...

/Users/sofiiapelenska/project/ML-tasks/Task 2/testTask2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/sofiiapelenska/project/ML-tasks/Task 2/testTask2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model evaluation

In [4]:
metrics = trainer.predict(trainer.eval_dataset).metrics

for k, v in metrics.items():
    print(f"{k}: {v}")

/Users/sofiiapelenska/project/ML-tasks/Task 2/testTask2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


/Users/sofiiapelenska/project/ML-tasks/Task 2/testTask2/lib/python3.12/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ANIMAL seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


test_loss: 0.00047970065497793257
test_f1: 1.0
test_runtime: 1.2093
test_samples_per_second: 35.558
test_steps_per_second: 2.481


<h1 align="center">Image Classification (ResNet18)</h1>

At this stage, a **ResNet18-based convolutional neural network** is used to classify animals in images.

ResNet18 was chosen because it is a compact and efficient deep CNN architecture that provides strong performance while remaining relatively fast to train and evaluate.

In this project, ResNet18 is trained from scratch for image classification on the Animals10 dataset.

The model learns to recognize the following animal classes:

- **cat**
- **dog**
- **cow**
- **horse**
- **sheep**
- **elephant**
- **butterfly**
- **chicken**
- **spider**
- **squirrel**

During training, images are resized and normalized before being passed through the network, allowing the model to learn visual features that distinguish different animal classes.

<h2 align="center">Dataset Setup</h2>

This project uses the **Animals10** dataset:  
https://www.kaggle.com/datasets/alessiocorrado99/animals10

The dataset must be located at:

data/cv/raw-img/


---

### Option 1 — Automatic download using Kaggle API

1. Go to https://www.kaggle.com/account  
2. Scroll to **API**  
3. Click **Create New API Token**  
4. Download `kaggle.json`

Place the file in:

~/.kaggle/kaggle.json

In [1]:
from src.utils.dataset_loader import download_animals10

download_animals10()

Dataset URL: https://www.kaggle.com/datasets/alessiocorrado99/animals10
License(s): GPL-2.0
animals10.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset ready.



---

### Option 2 — Manual Dataset Download

1. Open the dataset page:  
   https://www.kaggle.com/datasets/alessiocorrado99/animals10

2. Click **Download** and save the archive (`animals10.zip`).

3. Extract the archive.

4. Move the **raw-img** folder to the following location in the project:  data/cv/raw-img/

The final folder structure should look like this:

<img src="data_structure.jpg" width="200">


Start model training

In [2]:
from src.cv.train import train_model
from src.utils.labels import idx_to_english

classes = train_model("data/cv/raw-img", epochs=5)

idx_to_class = idx_to_english(classes)

Using device: mps


Epoch 1/5: 100%|██████████| 655/655 [02:08<00:00,  5.11it/s]


Train Loss: 1.7876 | Train Acc: 0.434 | Val Acc: 0.553


Epoch 2/5: 100%|██████████| 655/655 [02:07<00:00,  5.15it/s]


Train Loss: 1.4094 | Train Acc: 0.606 | Val Acc: 0.623


...

Train Loss: 1.2458 | Train Acc: 0.683 | Val Acc: 0.674


Epoch 4/5: 100%|██████████| 655/655 [02:07<00:00,  5.15it/s]


Train Loss: 1.1110 | Train Acc: 0.751 | Val Acc: 0.666


Epoch 5/5: 100%|██████████| 655/655 [02:07<00:00,  5.13it/s]


Train Loss: 1.0111 | Train Acc: 0.798 | Val Acc: 0.777
Model saved to: models/cv_model/resnet18.pth


Model evaluation

In [3]:
from src.cv.train import build_dataloaders, evaluate
from src.cv.inference import load_model
from src.utils.device import get_device

train_loader, val_loader, _ = build_dataloaders("data/cv/raw-img")

device = get_device()

model = load_model("models/cv_model/resnet18.pth")
acc = evaluate(model, val_loader, device)

print("Validation accuracy:", acc)

Using device: mps
Using device: mps
Validation accuracy: 0.8454632282712512


> **Note:**  
> The model was trained for only 5 epochs to reduce training time in the demo notebook.  
> Increasing the number of epochs would likely lead to higher validation accuracy, as the model would have more time to learn from the dataset.

<h1 align="center">Animal Detection Pipeline</h1>

The final stage of the project combines the NER model and the image classification model into a unified **AnimalPipeline**.

The pipeline integrates both components to process text and images together:

In [1]:
from src.pipeline.pipeline import AnimalPipeline

pipeline = AnimalPipeline()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

To demonstrate the functionality of the full pipeline, an interactive demo interface is provided.

In [7]:
!jupyter labextension enable widgetsnbextension

Exapmle:

In [3]:
from src.utils.demo_ui import interactive_demo

interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()

<h1 align="center">Edge Cases</h1>

In this example, the text does not contain any animal entity while the image classifier predicts squirrel, resulting in Match: False

In [11]:
interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()

In this edge case, the text mentions cow while the image classifier predicts dog, so the pipeline correctly detects a mismatch and returns Match: False.

In [ ]:
interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()

In this edge case, the text mentions two animals (cat and elephant), but the image classifier predicts only elephant, so the pipeline returns Match: False because not all animals mentioned in the text are present in the image.

In [4]:
interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()

In this edge case, the text mentions cow, but the image contains no visible animals and the classifier predicts spider, resulting in Match: False.  
This happens because the model was trained only on animal classes and therefore still predicts the most probable class even when no animal is present. A possible solution is to add a "no animal" class or introduce a confidence threshold to detect images without animals.

In [13]:
interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()

In this edge case, the text contains the plural form "elephants", while the image classifier predicts "elephant", resulting in Match: False. This happens because the NER model extracts the word exactly as it appears in the text, while the classifier outputs the singular class name. A possible solution is to normalize the extracted entities by converting plural forms to their base form (e.g., using lemmatization or a simple mapping).

In [5]:
interactive_demo(pipeline)

HTML(value='<b>Possible classes:</b> dog, horse, elephant, butterfly, chicken, cat, cow, sheep, spider, squirr…

Text(value='There is a cow in the picture', description='Sentence:', layout=Layout(width='500px'))

FileUpload(value=(), accept='image/*', description='Upload image')

Button(description='Run pipeline', style=ButtonStyle())

Output()